In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings

from scipy.cluster.hierarchy import linkage, dendrogram

from sklearn.preprocessing import PolynomialFeatures, StandardScaler, LabelEncoder, MinMaxScaler, RobustScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, SGDClassifier
from sklearn.svm import SVR, SVC
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import (mean_squared_error, mean_absolute_error, root_mean_squared_error, 
    r2_score, accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, silhouette_score, davies_bouldin_score)
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances, euclidean_distances

from imblearn.under_sampling import TomekLinks
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

from statsmodels.stats.stattools import jarque_bera, omni_normtest, durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

import statsmodels.api as sm

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import re
import nltk
import codecs

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

import chardet

from gensim.models import Word2Vec

import multiprocessing

import os
import sys

sys.path.append(os.getcwd() + '/numi_libs/')

import file_io as fio
import data_eda as eda
import data_forge as frg
import data_model as dmd

nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')

warnings.filterwarnings('ignore')

seed = 420
np.random.seed(seed)

In [ ]:
filename_c = "patient-data.csv"
filename_n = 'curse-of-dimensionality.xlsx'

df_c = fio.readCsvToDF(filename_c)

eda.initialAssessment(df_c, "Patient Dataset")


file_path = 'curse-of-dimensionality.xlsx'
df_n = pd.read_excel(filename_n, sheet_name='Sheet3')

eda.initialAssessment(df_n, "Curse of Dimensionality Dataset")
print("assessment complete")

In [ ]:
eda.generateEDATables(df_c)

In [ ]:
#eda.performPcaAnalysis(df_c.drop(columns = ['Ailment']))

In [ ]:
df_c = frg.simpleImputation(df_c)

x = df_c.drop(columns = ['Ailment'])
y = df_c['Ailment']

x, y = frg.smoteennSample(x, y, {
    "Heart Disease": 500,
    "Thromboc": 500
})

"""
x, y = frg.tomekSample(x, y, [
    "Heart Disease",
    "Thromboc"
])
"""

df_c0 = x
df_c0["Ailment"] = y
eda.generateEDATables(df_c0)

In [ ]:
eda.performPcaAnalysis(df_c0.drop(columns = ['Ailment']))

In [ ]:
algorithms = {
    #'Linear Classifier': SVC(kernel = "linear", probability = True),
    'Logistic Regression': LogisticRegression(max_iter = 2000), 
    #'RandomForest': RandomForestClassifier(max_depth = 5, min_samples_leaf = 5),
    #'XGBoost': GradientBoostingClassifier(),
    #'knn': KNeighborsClassifier(),
    #'Neural Network-10, 10': MLPClassifier(hidden_layer_sizes=[10, 10], max_iter=5000),
}

pca = []

x_tr, x_te, y_tr, y_te = frg.ttSplit(df_c, "Ailment", True)

dmd.applyAndReportClassifiers(x_tr, x_te, y_tr, y_te, algorithms, pca)

In [ ]:
x = df_c.drop(columns = ['Ailment'])
y = df_c['Ailment']

x, y = frg.smoteennSample(x, y, {
    "Heart Disease": 500,
    "Thromboc": 500
})

"""
x, y = frg.tomekSample(x, y, [
    "Heart Disease",
    "Thromboc"
])
"""

df_c0 = x
df_c0["Ailment"] = y

x_tr, x_te, y_tr, y_te = frg.ttSplit(df_c0, "Ailment", True)

dmd.applyAndReportClassifiers(x_tr, x_te, y_tr, y_te, algorithms, pca)

In [ ]:
#eda.profileFeatures(df_c0)

In [ ]:
x_tr, x_te, _= frg.minMaxScale(x_tr, x_te)

dmd.applyAndReportClassifiers(x_tr, x_te, y_tr, y_te, algorithms, pca, "MinMaxed")